# 00 - Construccion del dataset

Este notebook descarga los precios, construye las variables de entrada y guarda el dataset oficial dentro de `data/`. Si ya existen `dataset_tfg.pkl` y `dataset_tfg.csv`, no hace falta ejecutarlo salvo que se quiera reconstruir el dataset desde cero.


## Definicion de la descarga y construccion

Esta celda fija los activos, pesos de la cartera, rango temporal y funciones para descargar precios y construir el dataset con lags, volatilidades y target.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import yfinance as yf


ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
DATA = ROOT / "data"
DATA.mkdir(exist_ok=True)

START_DATE = "2010-01-01"
END_DATE = "2024-12-31"

TICKERS = {
    "SP500": "^GSPC",
    "Gold": "GC=F",
    "Oil": "CL=F",
    "EURUSD": "EURUSD=X",
    "IEF": "IEF",
}

WEIGHTS = np.array([0.2] * 5)
N_LAGS = 15


def download_prices(start_date: str = START_DATE, end_date: str = END_DATE) -> pd.DataFrame:
    raw = yf.download(
        list(TICKERS.values()),
        start=start_date,
        end=end_date,
        auto_adjust=True,
        progress=False,
        threads=False,
    )
    prices = raw["Close"].copy()
    prices.columns = TICKERS.keys()
    return prices.dropna()


def build_dataset(prices: pd.DataFrame) -> pd.DataFrame:
    returns = np.log(prices / prices.shift(1)).dropna()
    portfolio_returns = returns @ WEIGHTS
    loss = -portfolio_returns
    target = loss.shift(-1)

    lagged_returns = pd.concat([portfolio_returns.shift(i) for i in range(N_LAGS)], axis=1)
    lagged_returns.columns = [f"rp_lag_{i}" for i in range(N_LAGS)]

    vol_20 = portfolio_returns.rolling(20).std(ddof=0).rename("vol_20")
    vol_60 = portfolio_returns.rolling(60).std(ddof=0).rename("vol_60")

    returns_renamed = returns.copy()
    returns_renamed.columns = [f"{col}_ret" for col in returns.columns]

    x = pd.concat([lagged_returns, vol_20, vol_60, returns_renamed], axis=1)
    dataset = pd.concat([x, target.rename("target")], axis=1).dropna()
    return dataset


def main() -> None:
    prices = download_prices()
    dataset = build_dataset(prices)

    prices.to_csv(DATA / "prices_2010_2024.csv")
    dataset.to_pickle(DATA / "dataset_tfg.pkl")
    dataset.to_csv(DATA / "dataset_tfg.csv")

    print("Prices:", prices.shape, prices.index.min().date(), prices.index.max().date())
    print("Dataset:", dataset.shape, dataset.index.min().date(), dataset.index.max().date())
    print("Saved dataset to", DATA / "dataset_tfg.pkl")


## Generacion del dataset oficial

Esta celda descarga los precios, construye el dataset y guarda los archivos `dataset_tfg.pkl` y `dataset_tfg.csv` en `data/`.


In [ ]:
prices = download_prices()
dataset = build_dataset(prices)

prices.to_csv(DATA / "prices_2010_2024.csv")
dataset.to_pickle(DATA / "dataset_tfg.pkl")
dataset.to_csv(DATA / "dataset_tfg.csv")

print("Prices:", prices.shape, prices.index.min().date(), prices.index.max().date())
print("Dataset:", dataset.shape, dataset.index.min().date(), dataset.index.max().date())
print("Guardado:", DATA / "dataset_tfg.pkl")
display(dataset.head())
